# Camada Gold — Data Marts e KPIs de Negócio (Olist)

Neste notebook construímos os Data Marts analíticos a partir das tabelas Silver e usamos `display()` para responder as perguntas de negócio.

São 2 projetos:
1. **Visão Comercial e Volume de Produtos** — receitas, rankings de mais/menos vendidos
2. **Satisfação de Clientes e Qualidade de Parceiros** — avaliações, rankings de melhores/piores produtos e vendedores

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalogo = "medalhao"
gold_db = "gold"

In [0]:
fat_pedidos         = spark.table(f"{catalogo}.silver.fat_pedidos")
fat_itens_pedidos   = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
fat_pagamentos      = spark.table(f"{catalogo}.silver.fat_pagamentos_pedidos")
fat_avaliacoes      = spark.table(f"{catalogo}.silver.fat_avaliacoes_pedidos")
fat_pedido_total    = spark.table(f"{catalogo}.silver.fat_pedido_total")
dim_produtos        = spark.table(f"{catalogo}.silver.dim_produtos")
dim_vendedores      = spark.table(f"{catalogo}.silver.dim_vendedores")
dim_consumidores    = spark.table(f"{catalogo}.silver.dim_consumidores")
dim_cotacao         = spark.table(f"{catalogo}.silver.dim_cotacao_dolar")
dim_cat_traducao    = spark.table(f"{catalogo}.silver.dim_categoria_produtos_traducao")

print("Tabelas Silver carregadas com sucesso!")

---
## 1º Projeto: Visão Comercial e Volume de Produtos

---
### Entrega 1 — gold.fat_vendas_comercial
Granularidade: Ano, Mês e Categoria de Produto.

Métricas: total_pedidos (distinct), qtd_itens_vendidos, receita_total_brl, receita_total_usd, ticket_medio_brl. Valores financeiros arredondados para 2 casas decimais.

In [0]:
itens_com_pedido = (
    fat_itens_pedidos
    .join(fat_pedidos.select("id_pedido", "data_pedido"), on="id_pedido", how="inner")
    .join(dim_produtos.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .join(
        dim_cotacao.select(
            F.col("data_cotacao"),
            F.col("cotacao_compra")
        ),
        F.to_date(F.col("data_pedido")) == F.col("data_cotacao"),
        how="left"
    )
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
)

fat_vendas_comercial = (
    itens_com_pedido
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        F.round(F.sum(F.col("preco_BRL") / F.col("cotacao_compra")), 2).alias("receita_total_usd")
    )
    .withColumn(
        "ticket_medio_brl",
        F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2)
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

fat_vendas_comercial.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{gold_db}.fat_vendas_comercial")

print(f"✅ fat_vendas_comercial: {fat_vendas_comercial.count()} registros")
fat_vendas_comercial.display()

### Entrega 2 — Rankings Comerciais

Top 5 produtos **mais** vendidos e Top 5 **menos** vendidos (por quantidade vendida), exibindo id do produto, categoria e quantidade.

In [0]:
vendas_por_produto = (
    fat_itens_pedidos
    .join(dim_produtos.select("id_produto","nome_produto", "categoria_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(F.count("id_item").alias("qtd_vendida"))
)

In [0]:
print("=== TOP 5 PRODUTOS MAIS VENDIDOS ===")
top5_mais = vendas_por_produto.orderBy(F.col("qtd_vendida").desc()).limit(5).drop("id_produto")
top5_mais.display()

In [0]:
print("=== TOP 5 PRODUTOS MENOS VENDIDOS ===")
top5_menos = vendas_por_produto.orderBy(F.col("qtd_vendida").asc()).limit(5).drop("id_produto")
top5_menos.display()

---
## 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros

---
### Entrega 1 — gold.fat_avaliacoes_clientes
Granularidade: Categoria do Produto, ID do Vendedor e Estado do Vendedor.

Métricas: total_avaliacoes, avaliacao_media, total_avaliacoes_positivas (nota >= 4), total_avaliacoes_negativas (nota <= 2), percentual_satisfacao.

In [0]:
avaliacoes_base = (
    fat_avaliacoes
        .join(fat_itens_pedidos.select("id_pedido", "id_produto", "id_vendedor"), on="id_pedido", how="inner")
        .join(dim_produtos.select("id_produto", "nome_produto", "categoria_produto"), on="id_produto", how="left")
        .join(dim_vendedores.select("id_vendedor", "nome_vendedor", "estado"), on="id_vendedor", how="left")
)

fat_avaliacoes_clientes = (
    avaliacoes_base
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    .withColumn(
        "percentual_satisfacao",
        F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2)
    )
)

fat_avaliacoes_clientes.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{gold_db}.fat_avaliacoes_clientes")

print(f"✅ fat_avaliacoes_clientes: {fat_avaliacoes_clientes.count()} registros")
fat_avaliacoes_clientes.display()

### Entrega 2 — Rankings de Qualidade

Critério de ordenação composto: primeiro pela nota (maior ou menor), usando o volume de avaliações de forma decrescente como desempate.

Exibir:
1. Produto MAIS bem avaliado
2. Produto MENOS bem avaliado
3. Vendedor MAIS bem avaliado
4. Vendedor MENOS bem avaliado

In [0]:
ranking_produtos = (
    avaliacoes_base
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
    )
)

In [0]:
print("=== PRODUTO MAIS BEM AVALIADO ===")
ranking_produtos.orderBy(
    F.col("avaliacao_media").desc(),
    F.col("total_avaliacoes").desc()
).limit(1).drop("id_produto").display()

In [0]:
print("=== PRODUTO MENOS BEM AVALIADO ===")
ranking_produtos.orderBy(
    F.col("avaliacao_media").asc(),
    F.col("total_avaliacoes").desc()
).limit(1).drop("id_produto").display()

In [0]:
ranking_vendedores = (
    avaliacoes_base
    .groupBy("id_vendedor", "nome_vendedor", "estado")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
    )
)

In [0]:
print("=== VENDEDOR MAIS BEM AVALIADO ===")
ranking_vendedores.orderBy(
    F.col("avaliacao_media").desc(),
    F.col("total_avaliacoes").desc()
).limit(1).drop("id_vendedor").display()

In [0]:
print("=== VENDEDOR MENOS BEM AVALIADO ===")
ranking_vendedores.orderBy(
    F.col("avaliacao_media").asc(),
    F.col("total_avaliacoes").desc()
).limit(1).drop("id_vendedor").display()

---
### Validação — Tabelas criadas na camada Gold

In [0]:
%sql

SHOW TABLES IN medalhao.gold;

In [0]:
for tabela in ["fat_vendas_comercial", "fat_avaliacoes_clientes"]:
    qtd = spark.table(f"{catalogo}.{gold_db}.{tabela}").count()
    print(f"{tabela}: {qtd} registros")